In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/nova-gbm-preds/nova_gbm_preds.csv
/kaggle/input/datasets/raesiow/nova-distilbert-oof/__huggingface_repos__.json
/kaggle/input/datasets/raesiow/nova-distilbert-oof/distilbert_oof.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/baseline_preds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_text_vs_both.png
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_tfidf.png
/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv


In [2]:
import os  # for searching folders
import pandas as pd  # tables

paths = {}  # file name -> full path
for folder, _, files in os.walk("/kaggle/input"):  # look through every input folder
    for name in files:  # each file
        if name in ["fpx_off_folds.csv", "distilbert_oof.csv", "nova_gbm_preds.csv"]:  # the three we need
            paths[name] = os.path.join(folder, name)  # remember where it is
print(paths)  # should list all three

folds = pd.read_csv(paths["fpx_off_folds.csv"])  # matched NOVA rows with text, label and fold
oof = pd.read_csv(paths["distilbert_oof.csv"])  # current DistilBERT out-of-fold probabilities
gbm = pd.read_csv(paths["nova_gbm_preds.csv"])  # current GBM predictions with OFF nutrients
for name, table in [("folds", folds), ("oof", oof), ("gbm", gbm)]:  # each table
    print(name, table.shape)  # rows and columns
    print(list(table.columns))  # column names
print(gbm["source"].value_counts())  # expect both 1030, mini_only 711, rf_only 143

{'nova_gbm_preds.csv': '/kaggle/input/datasets/raesiow/nova-gbm-preds/nova_gbm_preds.csv', 'distilbert_oof.csv': '/kaggle/input/datasets/raesiow/nova-distilbert-oof/distilbert_oof.csv', 'fpx_off_folds.csv': '/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv'}
folds (1884, 22)
['fpx_code', 'fpx_name', 'nova', 'off_code', 'off_name', 'source', 'score', 'cos', 'protein_g_f', 'fat_g_f', 'carb_excl_fiber_g_f', 'sugars_g_f', 'fiber_g_f', 'sat_fat_g_f', 'sodium_g_f', 'ingredients_text', 'categories_en', 'pnns_groups_2', 'image_url', 'nova_group', 'group', 'fold']
oof (1884, 8)
['fpx_code', 'off_code', 'nova', 'fold', 'p1', 'p2', 'p3', 'p4']
gbm (1884, 37)
['fpx_code', 'fpx_name', 'nova', 'off_code', 'off_name', 'source', 'score', 'cos', 'protein_g_f', 'fat_g_f', 'carb_excl_fiber_g_f', 'sugars_g_f', 'fiber_g_f', 'sat_fat_g_f', 'sodium_g_f', 'ingredients_text', 'categories_en', 'pnns_groups_2', 'image_url', 'nova_group', 'group', 'fold', 'p1', 'p2', 'p3', 'p4', 'protein_g_off', 'fat_

In [3]:
import numpy as np  # arrays
import torch  # deep learning library
from torch.utils.data import DataLoader, TensorDataset  # feed data in batches
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup  # DistilBERT parts

device = "cuda" if torch.cuda.is_available() else "cpu"  # use the GPU
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")  # turns text into word pieces
MAX_LEN, EPOCHS, BATCH, LR = 256, 10, 16, 2e-5  # same settings as the milestone model

def encode(texts):  # text list -> numbers DistilBERT understands
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")  # cut at 256, pad the rest

def train_distilbert(texts, labels):  # train one DistilBERT on these texts and NOVA labels (1-4)
    torch.manual_seed(42)  # same random start every time
    y = torch.tensor(np.array(labels) - 1)  # labels 1-4 -> 0-3
    enc = encode(texts)  # texts -> numbers
    loader = DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"], y), batch_size=BATCH, shuffle=True)  # batches of 16
    counts = np.bincount(y.numpy(), minlength=4)  # how many rows per NOVA group
    weights = torch.tensor(len(y) / (4 * np.maximum(counts, 1)), dtype=torch.float).to(device)  # balanced class weights
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)  # fresh model
    opt = torch.optim.AdamW(model.parameters(), lr=LR)  # optimiser (default weight decay 0.01)
    steps = EPOCHS * len(loader)  # total training steps
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)  # 10% warm-up, then slowly lower the learning rate
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)  # weighted loss for rare groups
    model.train()  # training mode
    for epoch in range(EPOCHS):  # 10 passes over the data
        for ids, mask, yb in loader:  # each batch
            ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)  # move to GPU
            loss = loss_fn(model(input_ids=ids, attention_mask=mask).logits, yb)  # how wrong the model is
            opt.zero_grad()  # clear old gradients
            loss.backward()  # work out new gradients
            opt.step()  # update the model
            sched.step()  # update the learning rate
    return model  # the trained model

def predict_probs(model, texts):  # texts -> probabilities for NOVA 1-4
    model.eval()  # evaluation mode
    enc = encode(texts)  # texts -> numbers
    out = []  # collect results
    with torch.no_grad():  # no training here
        for i in range(0, len(texts), 64):  # 64 texts at a time
            ids = enc["input_ids"][i:i + 64].to(device)  # this batch's word pieces
            mask = enc["attention_mask"][i:i + 64].to(device)  # which pieces are real text
            out.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=1).cpu().numpy())  # probabilities
    return np.vstack(out)  # one row of 4 probabilities per text
print("ready")  # done

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

ready


In [5]:
import time  # timing
data = gbm.copy()  # work on a copy of the table
data["ingredients_text"] = data["ingredients_text"].fillna("")  # empty text instead of missing
new_probs = np.zeros((len(data), 4))  # space for the new probabilities
start = time.time()  # start the clock
for k in sorted(data["fold"].unique()):  # each of the 5 folds
    train_rows = (data["fold"] != k) & (data["source"] == "both")  # train: other folds, both-method pairs only
    test_rows = data["fold"] == k  # predict: every row of this fold
    model = train_distilbert(data.loc[train_rows, "ingredients_text"], data.loc[train_rows, "nova"])  # train
    new_probs[test_rows.values] = predict_probs(model, data.loc[test_rows, "ingredients_text"])  # predict this fold
    del model  # free the model
    torch.cuda.empty_cache()  # free GPU memory
    print(f"fold {k} done: trained on {train_rows.sum()} rows, {time.time() - start:.0f} s so far")  # progress
for j in range(4):  # NOVA 1-4
    data[f"q{j + 1}"] = new_probs[:, j]  # new probabilities as columns q1-q4
data.to_csv("/kaggle/working/both_pairs_oof.csv", index=False)  # save straight away
print("saved")  # done

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 0 done: trained on 830 rows, 196 s so far


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

... (7 outputs left out)

fold 3 done: trained on 834 rows, 767 s so far


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 4 done: trained on 836 rows, 962 s so far
saved


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier  # the same booster as the milestone model
from sklearn.metrics import f1_score  # macro-F1

nutrients = ["protein_g_off", "fat_g_off", "carb_excl_fiber_g_off", "sugars_g_off",  # the 7 OFF nutrients
             "fiber_g_off", "sat_fat_g_off", "sodium_g_off"]
features_b = ["q1", "q2", "q3", "q4"] + nutrients  # new probabilities + nutrients

rows = []  # one result row per fold
for k in sorted(data["fold"].unique()):  # each fold
    train_b = (data["fold"] != k) & (data["source"] == "both")  # B trains on both-method rows of other folds
    test = (data["fold"] == k) & (data["source"] == "both")  # BOTH models are scored on these same rows
    gbm_b = HistGradientBoostingClassifier(class_weight="balanced", max_iter=200,  # same settings as milestone
                                           learning_rate=0.05, random_state=42)
    gbm_b.fit(data.loc[train_b, features_b], data.loc[train_b, "nova"])  # train B
    pred_b = gbm_b.predict(data.loc[test, features_b])  # B's predictions
    f1_a = f1_score(data.loc[test, "nova"], data.loc[test, "pred_bert_off"], average="macro")  # A = current app model
    f1_b = f1_score(data.loc[test, "nova"], pred_b, average="macro")  # B = both-method model
    rows.append({"fold": k, "test_rows": int(test.sum()), "A_current": round(f1_a, 3), "B_both": round(f1_b, 3)})  # store

result = pd.DataFrame(rows)  # results table
print(result)  # per fold
print("mean A:", round(result["A_current"].mean(), 3), "+/-", round(result["A_current"].std(), 3))  # current model
print("mean B:", round(result["B_both"].mean(), 3), "+/-", round(result["B_both"].std(), 3))  # both-method model
wins = int((result["B_both"] > result["A_current"]).sum())  # folds where B is better
print("B better in", wins, "of 5 folds ->", "ADOPT B" if wins >= 4 else "KEEP current model")  # the rule fixed beforehand
result.to_csv("/kaggle/working/both_pairs_comparison.csv", index=False)  # save

   fold  test_rows  A_current  B_both
0     0        200      0.826   0.804
1     1        201      0.836   0.838
2     2        239      0.675   0.808
3     3        196      0.705   0.811
4     4        194      0.729   0.777
mean A: 0.754 +/- 0.073
mean B: 0.808 +/- 0.022
B better in 4 of 5 folds -> ADOPT B


In [7]:
final_rows = data["source"] == "both"  # all both-method pairs, no folds: this is the final model
final_bert = train_distilbert(data.loc[final_rows, "ingredients_text"], data.loc[final_rows, "nova"])  # final DistilBERT
final_gbm = HistGradientBoostingClassifier(class_weight="balanced", max_iter=200,  # same settings as before
                                           learning_rate=0.05, random_state=42)
final_gbm.fit(data.loc[final_rows, features_b], data.loc[final_rows, "nova"])  # booster on out-of-fold probs + nutrients
print("final models trained on", int(final_rows.sum()), "rows")  # expect 1030

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


final models trained on 1030 rows


In [9]:
cat_path, off_path = None, None  # file locations, not found yet
for folder, _, files in os.walk("/kaggle/input"):  # look through every input folder
    if "catalogue_predictions_v2.csv" in files:  # the app's catalogue
        cat_path = os.path.join(folder, "catalogue_predictions_v2.csv")  # remember it
    if "off_filtered_all_countries.csv" in files:  # cleaned OFF with ingredients and nutrients
        off_path = os.path.join(folder, "off_filtered_all_countries.csv")  # remember it
print(cat_path, off_path)  # both should be paths, not None

cat = pd.read_csv(cat_path, dtype={"code": str})  # the 71,298 catalogue products
off = pd.read_csv(off_path, dtype={"code": str}, usecols=["code", "ingredients_text", "protein_g", "fat_g",  # text and the 7 nutrients
                  "carb_excl_fiber_g", "sugars_g", "fiber_g", "sat_fat_g", "sodium_g"])
merged = cat[["code"]].merge(off, on="code", how="left")  # same order as the catalogue
print("rows:", len(merged), "| missing text:", merged["ingredients_text"].isna().sum())  # expect 71298 and about 0

probs = predict_probs(final_bert, merged["ingredients_text"].fillna(""))  # DistilBERT probabilities for every product
X = pd.DataFrame(probs, columns=["q1", "q2", "q3", "q4"])  # probabilities as columns
for n in nutrients:  # each nutrient, named *_off in training
    X[n] = merged[n.replace("_off", "")].values  # same nutrient from the catalogue
gbm_probs = final_gbm.predict_proba(X[features_b])  # booster probabilities for NOVA 1-4

cat["nova_pred_v2"] = cat["nova_pred"]  # keep the old prediction for comparison
cat["nova_confidence_v2"] = cat["nova_confidence"]  # and its confidence
cat["nova_pred"] = final_gbm.classes_[gbm_probs.argmax(axis=1)]  # new NOVA group
cat["nova_confidence"] = gbm_probs.max(axis=1)  # new confidence = top probability
print(pd.crosstab(cat["nova_pred_v2"], cat["nova_pred"]))  # old NOVA (rows) vs new NOVA (columns)
cat.to_csv("/kaggle/working/catalogue_predictions_v3.csv", index=False)  # save
print("saved v3")  # done

/kaggle/input/datasets/raesiow/catalogue-predictions-v2/catalogue_predictions_v2.csv /kaggle/input/datasets/raesiow/cleaned-off/off_filtered_all_countries.csv
rows: 71298 | missing text: 0
nova_pred        1     2     3      4
nova_pred_v2                         
1             9365    60   823   1323
2              157  1425    47     47
3              774    42  5952   2089
4             1629   274  1430  45861
saved v3
